In [3]:
from collections import deque


def read_input_file(filename):
    with open(filename, "r") as file:
        lines = [line.strip() for line in file if line.strip()]

    # Line 1: vertices with colors, example: 1w 2w 3r 4b
    vertex_tokens = lines[0].split()

    vertex_names = []
    colors = {}

    for token in vertex_tokens:
        vertex_names.append(token)      # keep full name like "1b"
        colors[token] = token[-1] 

    n = len(vertex_names)

    # Next n lines: adjacency matrix
    adj_matrix = []
    for i in range(1, n + 1):
        row = list(map(int, lines[i].split()))
        adj_matrix.append(row)

    # Next line: start and target
    s, t = lines[n + 1].split()

    # Last line: k
    k = int(lines[n + 2])

    return vertex_names, colors, adj_matrix, s, t, k


def is_valid_next_color(current_color, next_color):
    """
    Checks if the next color correctly follows the 3-color cycle.

    We allow:
    b -> w
    w -> r
    r -> b
    """
    valid_next = {
        "b": "w",
        "w": "r",
        "r": "b"
    }

    return valid_next[current_color] == next_color


def solve_3csp(vertex_names, colors, adj_matrix, s, t, k):
    n = len(vertex_names)

    # Convert vertex name to matrix index
    index_map = {}
    for i in range(n):
        index_map[vertex_names[i]] = i

    s_index = index_map[s]
    t_index = index_map[t]

    # Special case: start is already target
    if s == t:
        return True, [s]

    queue = deque()
    visited = [False] * n
    parent = [-1] * n
    distance = [-1] * n

    queue.append(s_index)
    visited[s_index] = True
    distance[s_index] = 0

    while queue:
        current = queue.popleft()

        # Do not go past length k
        if distance[current] == k:
            continue

        current_vertex = vertex_names[current]
        current_color = colors[current_vertex]

        for neighbor in range(n):
            if adj_matrix[current][neighbor] == 1 and not visited[neighbor]:

                neighbor_vertex = vertex_names[neighbor]
                neighbor_color = colors[neighbor_vertex]

                # Main 3CSP rule
                if not is_valid_next_color(current_color, neighbor_color):
                    continue

                visited[neighbor] = True
                parent[neighbor] = current
                distance[neighbor] = distance[current] + 1
                queue.append(neighbor)

    if not visited[t_index]:
        return False, None

    # Build the path from t back to s
    path = []
    current = t_index

    while current != -1:
        path.append(vertex_names[current])
        current = parent[current]

    path.reverse()

    return True, path


def main():
    filename = input("Enter input file name: ")

    try:
        vertex_names, colors, adj_matrix, s, t, k = read_input_file(filename)
        accepted, path = solve_3csp(vertex_names, colors, adj_matrix, s, t, k)

        if accepted:
            print("Accept")
            print("Path:", " -> ".join(path))
            print("Length:", len(path) - 1)

            color_path = []
            for vertex in path:
                color_path.append(colors[vertex])

            print("Colors:", " -> ".join(color_path))
        else:
            print("Reject")

    except FileNotFoundError:
        print("Error: File not found.")
    except Exception as e:
        print("Error:", e)


if __name__ == "__main__":
    main()

Enter input file name:  converted_3csp.txt


Accept
Path: 1b -> 1w -> 1r -> 2b -> 2w -> 2r -> 4b
Length: 6
Colors: b -> w -> r -> b -> w -> r -> b
